# 09 — Cross-family ensemble: CatBoost + MLP (neural net)
Every earlier model is a **tree** (CatBoost, RF, ExtraTrees, GB). They share an axis-aligned inductive bias, so their ensembles (`sub02` soft-vote, `sub04` CatBoost-diversity) stayed stuck at the ~0.829 wall — the members make the *same* mistakes. A neural net breaks that bias: a standardized **MLP** rivals/beats CatBoost on its own and, crucially, makes **different** errors. The real win is the **cross-family blend**.

Honesty guards (all motivated by an adversarial audit of this idea):
- **Fixed 0.5/0.5 weights**, chosen a-priori. Searching weights on the OOF inflated the score by ~0.006 of noise that did *not* transfer out-of-fold — so we don't tune weights.
- **Paired repeated-CV headline.** On the single seed-42 split CatBoost is ~+0.003 lucky, which hides the gain; the honest number is a paired `RepeatedStratifiedKFold(5×5)` vs single CatBoost on identical, leak-free folds.

In [1]:
# --- Shared toolbox (identical across notebooks; see build_notebooks.py) ---
import warnings, json
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
N_FOLDS = 5
CLASS_NAMES = {0: "Wake", 1: "Light", 2: "Deep", 3: "REM"}
CLASSES = np.array([0, 1, 2, 3])
EOG = "eog_burst_index"            # the only column with missing values (~50%)

RAW_FEATURES = [
    "eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
    "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power", "eeg_spectral_entropy",
    "eeg_spindle_density", "eeg_kcomplex_rate", "emg_chin_tone", "emg_tone_variance",
    "eog_movement_density", "eog_amplitude", "heart_rate_mean", "heart_rate_variability",
    "respiration_rate", "respiration_variability", "spo2_mean", "body_movement_index",
    EOG,
]
POWER = ["eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
         "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power"]

HERE = Path.cwd()
ART = HERE / "artifacts"; ART.mkdir(exist_ok=True)
SUB = HERE / "submissions"; SUB.mkdir(exist_ok=True)


def load_data():
    """Return (train_df, test_df). Features kept as-is (NaN preserved)."""
    tr = pd.read_csv(HERE / "train.csv")
    te = pd.read_csv(HERE / "test.csv")
    return tr, te


def macro_f1(y_true, y_pred):
    """Competition metric: macro-averaged F1 over the 4 classes."""
    return f1_score(y_true, y_pred, average="macro")


def per_class_f1(y_true, y_pred):
    f = f1_score(y_true, y_pred, average=None, labels=CLASSES)
    return {CLASS_NAMES[c]: round(float(f[i]), 4) for i, c in enumerate(CLASSES)}


def softplus(x):
    """Numerically stable log(1+exp(x)); strictly positive and monotonic.
    Used to turn z-scored band powers (~50% negative) into positive magnitudes
    so band ratios are well-defined instead of dividing by near-zero."""
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _aligned_proba(model, X):
    """predict_proba with columns aligned to CLASSES = [0,1,2,3]."""
    p = model.predict_proba(X)
    cls = list(np.asarray(model.classes_))
    idx = [cls.index(c) for c in CLASSES]
    return p[:, idx]


def run_oof(make_model, X, y, X_test, folds, needs_impute=False, use_eval_set=False):
    """Out-of-fold training under fixed folds.

    Returns (oof, test_p, oof_macro, fold_scores):
      oof     : (n_train, 4) out-of-fold probabilities (each row predicted once)
      test_p  : (n_test, 4) test probabilities, averaged over the 5 fold-models
      oof_macro: global macro-F1 over the assembled OOF matrix (primary metric)

    Two model families, identical folds:
      - CatBoost (needs_impute=False): NaN passed through natively.
      - sklearn trees (needs_impute=True): add EOG-missing flag + fill EOG NaN
        with the TRAIN-FOLD median (fit on train fold only -> no leakage)."""
    n = len(y)
    oof = np.zeros((n, len(CLASSES)))
    test_p = np.zeros((len(X_test), len(CLASSES)))
    fold_scores = []
    for tr_idx, va_idx in folds:
        Xtr, Xva, Xte = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy(), X_test.copy()
        ytr, yva = y[tr_idx], y[va_idx]
        if needs_impute:
            med = Xtr[EOG].median()
            for d in (Xtr, Xva, Xte):
                if EOG + "_missing" not in d.columns:
                    d[EOG + "_missing"] = d[EOG].isna().astype("int8")
                d[EOG] = d[EOG].fillna(med)
            assert not Xtr.isna().any().any(), "NaN remained after impute"
        model = make_model()
        if use_eval_set:
            model.fit(Xtr, ytr, eval_set=(Xva, yva))
        else:
            model.fit(Xtr, ytr)
        oof[va_idx] = _aligned_proba(model, Xva)
        test_p += _aligned_proba(model, Xte) / len(folds)
        fold_scores.append(macro_f1(yva, oof[va_idx].argmax(1)))
    oof_macro = macro_f1(y, oof.argmax(1))
    return oof, test_p, oof_macro, fold_scores


def load_folds():
    """Load the fixed StratifiedKFold split saved by 02_baseline."""
    d = np.load(ART / "folds.npz", allow_pickle=True)
    return [(d[f"tr{i}"], d[f"va{i}"]) for i in range(N_FOLDS)]


In [2]:
def log_result(step, model, feature_set, oof_macro, pcf, notes=""):
    """Write one row to results_log.csv. Idempotent per (step, model): re-running
    a notebook replaces its own row rather than duplicating it."""
    import csv
    path = HERE / "results_log.csv"
    header = ["step", "model", "feature_set", "oof_macro_f1",
              "f1_Wake", "f1_Light", "f1_Deep", "f1_REM", "notes"]
    rows = []
    if path.exists():
        with open(path, newline="") as fh:
            data = list(csv.reader(fh))
        if data and data[0] == header:
            rows = [r for r in data[1:] if not (len(r) >= 2 and r[0] == step and r[1] == model)]
    row = [step, model, feature_set, round(float(oof_macro), 5),
           pcf["Wake"], pcf["Light"], pcf["Deep"], pcf["REM"], notes]
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header); w.writerows(rows); w.writerow(row)
    print("logged:", step, model, "OOF macro-F1 =", round(float(oof_macro), 5))


In [3]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold
from catboost import CatBoostClassifier

cols = json.load(open(ART / "feature_cols.json"))["columns"]
Xtr = pd.DataFrame(np.load(ART / "features_v1_train.npy"), columns=cols)
Xte = pd.DataFrame(np.load(ART / "features_v1_test.npy"), columns=cols)
y = np.load(ART / "y_train.npy")
test_ids = np.load(ART / "test_ids.npy")
folds = load_folds()
gate = json.load(open(ART / "repeated_cv_gate.json"))
bp = json.load(open(ART / "catboost_best_params.json"))

# Anchor = the deployed single CatBoost (sub01): reuse its saved OOF/test probs.
cat_oof = np.load(ART / "catboost_tuned_oof.npy")
cat_test = np.load(ART / "catboost_tuned_test.npy")
anchor = macro_f1(y, cat_oof.argmax(1))
print("features_v1:", Xtr.shape[1], "cols | anchor CatBoost OOF macro-F1:", round(anchor, 5))
print("gate:", gate["mean"], "+/-", gate["std"], "| deployed CatBoost config:", bp)

features_v1: 28 cols | anchor CatBoost OOF macro-F1: 0.82898
gate: 0.82598 +/- 0.00169 | deployed CatBoost config: {'depth': 6, 'lr': 0.03, 'l2': 3.0, 'balanced': False}


## MLP under the shared folds (scaled, imputed, seed-bagged)
`MLP(256,128)`, ReLU, `alpha=1e-3`, early stopping. Per fold: `StandardScaler` + EOG median fit on the **train fold only**; bagged over 3 seeds to tame MLP variance. We also report the **error-correlation** with CatBoost — the lower it is, the more an ensemble can help.

In [4]:
# MLP under the SAME folds. Non-tree models need scaling and cannot take NaN, so
# per fold we fit a StandardScaler and the EOG median on the TRAIN fold only
# (no leakage), then bag over a few seeds (an MLP is higher-variance than CatBoost).
MLP_SEEDS = [42, 7, 13]

def make_mlp(seed):
    return MLPClassifier(hidden_layer_sizes=(256, 128), alpha=1e-3,
        activation="relu", solver="adam", batch_size=256, max_iter=400,
        early_stopping=True, n_iter_no_change=20, validation_fraction=0.1,
        random_state=seed)

def mlp_oof(Xdf, ydat, Xtest_df, the_folds, seeds):
    """Seed-bagged MLP OOF + test probabilities, scaler+impute fit per train fold."""
    ncl = len(CLASSES)
    oof = np.zeros((len(ydat), ncl)); test_p = np.zeros((len(Xtest_df), ncl))
    for tr_idx, va_idx in the_folds:
        Xt, Xv, Xe = Xdf.iloc[tr_idx].copy(), Xdf.iloc[va_idx].copy(), Xtest_df.copy()
        med = Xt[EOG].median()
        for d in (Xt, Xv, Xe):
            d[EOG] = d[EOG].fillna(med)
        sc = StandardScaler().fit(Xt.values)
        Xts, Xvs, Xes = sc.transform(Xt.values), sc.transform(Xv.values), sc.transform(Xe.values)
        for s in seeds:
            m = make_mlp(s); m.fit(Xts, ydat[tr_idx])
            oof[va_idx] += _aligned_proba(m, Xvs) / len(seeds)
            test_p += _aligned_proba(m, Xes) / (len(the_folds) * len(seeds))
    return oof, test_p

mlp_oof_p, mlp_test_p = mlp_oof(Xtr, y, Xte, folds, MLP_SEEDS)
mlp_macro = macro_f1(y, mlp_oof_p.argmax(1))
cat_wrong = (cat_oof.argmax(1) != y).astype(int)
mlp_wrong = (mlp_oof_p.argmax(1) != y).astype(int)
errcorr = float(np.corrcoef(cat_wrong, mlp_wrong)[0, 1])
print(f"MLP seed-bag (K={len(MLP_SEEDS)}) OOF macro-F1 = {mlp_macro:.5f}")
print("MLP per-class F1:", per_class_f1(y, mlp_oof_p.argmax(1)))
print(f"error-correlation MLP vs CatBoost = {errcorr:.3f}  (lower = more diverse; tree-vs-tree ~0.9)")
np.save(ART / "mlp_oof.npy", mlp_oof_p); np.save(ART / "mlp_test.npy", mlp_test_p)
log_result("09_mlp_ensemble", "mlp_seedbag", "features_v1", mlp_macro,
           per_class_f1(y, mlp_oof_p.argmax(1)),
           f"MLP(256,128) a=1e-3 K={len(MLP_SEEDS)}; errcorr_vs_cat={errcorr:.3f}")

MLP seed-bag (K=3) OOF macro-F1 = 0.83677
MLP per-class F1: {'Wake': 0.8626, 'Light': 0.8536, 'Deep': 0.7889, 'REM': 0.8419}
error-correlation MLP vs CatBoost = 0.680  (lower = more diverse; tree-vs-tree ~0.9)
logged: 09_mlp_ensemble mlp_seedbag OOF macro-F1 = 0.83677


## Fixed-weight cross-family ensemble (no OOF weight search)
Equal-weight average of the CatBoost and MLP probabilities. Reported on the seed-42 split for continuity with the other notebooks, but read the paired test below for the honest gain.

In [5]:
# Fixed equal weights chosen A-PRIORI -- NO OOF weight search. The audit showed
# tuning weights on the full OOF inflated the score by ~0.006 of noise that did
# not transfer out-of-fold, so equal weights are the honest, defensible choice.
W_CAT, W_MLP = 0.5, 0.5
ens_oof = W_CAT * cat_oof + W_MLP * mlp_oof_p
ens_test = W_CAT * cat_test + W_MLP * mlp_test_p
ens_macro = macro_f1(y, ens_oof.argmax(1))
beats = ens_macro > anchor + gate["std"]
print(f"anchor single CatBoost     = {anchor:.5f}")
print(f"MLP seed-bag               = {mlp_macro:.5f}")
print(f"ENSEMBLE 0.5*Cat + 0.5*MLP = {ens_macro:.5f}   (delta vs anchor {ens_macro - anchor:+.5f})")
print(f"gate 1-sigma = {gate['std']:.5f} -> clears anchor+1sigma on the seed-42 split?", bool(beats))
print("ensemble per-class F1:", per_class_f1(y, ens_oof.argmax(1)))
print("(on the seed-42 split CatBoost is itself ~+0.003 lucky, so this single-split delta")
print(" UNDERSTATES the true gain; the paired repeated-CV test below is the honest headline.)")
np.save(ART / "mlp_ensemble_oof.npy", ens_oof); np.save(ART / "mlp_ensemble_test.npy", ens_test)
json.dump({"weights": {"catboost": W_CAT, "mlp": W_MLP}, "mlp_seeds": MLP_SEEDS,
           "mlp_macro": round(float(mlp_macro), 5), "anchor": round(float(anchor), 5),
           "ensemble_macro": round(float(ens_macro), 5), "errcorr_vs_cat": round(errcorr, 3),
           "beats_gate_seed42": bool(beats)}, open(ART / "mlp_ensemble.json", "w"), indent=2)
log_result("09_mlp_ensemble", "catboost_plus_mlp", "features_v1", ens_macro,
           per_class_f1(y, ens_oof.argmax(1)),
           f"fixed 0.5/0.5 Cat+MLP; vs anchor {anchor:.5f} ({ens_macro - anchor:+.5f}); errcorr={errcorr:.3f}")

anchor single CatBoost     = 0.82898
MLP seed-bag               = 0.83677
ENSEMBLE 0.5*Cat + 0.5*MLP = 0.83711   (delta vs anchor +0.00813)
gate 1-sigma = 0.00169 -> clears anchor+1sigma on the seed-42 split? True
ensemble per-class F1: {'Wake': 0.8622, 'Light': 0.8556, 'Deep': 0.7895, 'REM': 0.8411}
(on the seed-42 split CatBoost is itself ~+0.003 lucky, so this single-split delta
 UNDERSTATES the true gain; the paired repeated-CV test below is the honest headline.)
logged: 09_mlp_ensemble catboost_plus_mlp OOF macro-F1 = 0.83711


## Honest headline — paired `RepeatedStratifiedKFold(5×5)` vs single CatBoost
Both arms run on identical fresh folds (seed 2026) and are strictly leak-free (CatBoost at fixed iterations, MLP scaler/impute inside the train fold). This is the number to trust: it averages out the single-split fold luck that makes the seed-42 delta look smaller than it is.

In [6]:
# Honest headline: a PAIRED comparison on fresh folds. On any single split
# CatBoost's fold luck can hide the gain, so we average over
# RepeatedStratifiedKFold(5x5, seed=2026) and compare the ensemble against single
# CatBoost on IDENTICAL folds. Both arms are leak-free: CatBoost uses FIXED
# iterations (no early-stopping peek at the valid fold); the MLP fits its
# scaler+impute strictly inside the train fold. Weights stay fixed 0.5/0.5.
N_REP = 5
PAIR_SEEDS = [42, 7]                         # 2-seed MLP bag inside the loop (cost control)
CAT_ITERS = 900                              # ~ deployed best_iter (~921); fixed => leak-free
rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REP, random_state=2026)
splits = list(rskf.split(np.zeros(len(y)), y))

def make_cat_fixed(seed=SEED):
    return CatBoostClassifier(loss_function="MultiClass", eval_metric="TotalF1:average=Macro",
        iterations=CAT_ITERS, learning_rate=bp["lr"], depth=bp["depth"], l2_leaf_reg=bp["l2"],
        random_seed=seed, allow_writing_files=False, thread_count=-1, verbose=False)

cat_sc, ens_sc = [], []
for r in range(N_REP):
    cat_o = np.zeros((len(y), len(CLASSES)))
    mlp_o = np.zeros((len(y), len(CLASSES)))
    for tr_idx, va_idx in splits[r * N_FOLDS:(r + 1) * N_FOLDS]:
        cb = make_cat_fixed(); cb.fit(Xtr.iloc[tr_idx], y[tr_idx])
        cat_o[va_idx] = _aligned_proba(cb, Xtr.iloc[va_idx])
        Xt, Xv = Xtr.iloc[tr_idx].copy(), Xtr.iloc[va_idx].copy()
        med = Xt[EOG].median(); Xt[EOG] = Xt[EOG].fillna(med); Xv[EOG] = Xv[EOG].fillna(med)
        sc = StandardScaler().fit(Xt.values)
        Xts, Xvs = sc.transform(Xt.values), sc.transform(Xv.values)
        for s in PAIR_SEEDS:
            m = make_mlp(s); m.fit(Xts, y[tr_idx])
            mlp_o[va_idx] += _aligned_proba(m, Xvs) / len(PAIR_SEEDS)
    cs = macro_f1(y, cat_o.argmax(1))
    es = macro_f1(y, (0.5 * cat_o + 0.5 * mlp_o).argmax(1))
    cat_sc.append(cs); ens_sc.append(es)
    print(f"  repeat {r + 1}: CatBoost {cs:.5f} | Ensemble {es:.5f} | delta {es - cs:+.5f}")

cat_sc, ens_sc = np.array(cat_sc), np.array(ens_sc)
d = ens_sc - cat_sc
print(f"\nPAIRED repeated CV ({N_REP}x{N_FOLDS}, seed=2026):")
print(f"  single CatBoost : {cat_sc.mean():.5f} +/- {cat_sc.std():.5f}")
print(f"  Cat+MLP ensemble: {ens_sc.mean():.5f} +/- {ens_sc.std():.5f}")
print(f"  paired delta    : {d.mean():+.5f} +/- {d.std():.5f}  (positive in {int((d > 0).sum())}/{N_REP} repeats)")
print(f"  => honest gain beyond the {gate['std']:.5f} noise floor?", bool(d.mean() > gate["std"]))
json.dump({"n_repeats": N_REP, "n_folds": N_FOLDS, "cat_iters": CAT_ITERS,
           "pair_mlp_seeds": PAIR_SEEDS, "cat_mean": round(float(cat_sc.mean()), 5),
           "cat_std": round(float(cat_sc.std()), 5), "ens_mean": round(float(ens_sc.mean()), 5),
           "ens_std": round(float(ens_sc.std()), 5), "delta_mean": round(float(d.mean()), 5),
           "delta_std": round(float(d.std()), 5), "pos_repeats": int((d > 0).sum())},
          open(ART / "mlp_ensemble_paired_cv.json", "w"), indent=2)
log_result("09_mlp_ensemble", "catboost_plus_mlp_pairedCV", "features_v1", float(ens_sc.mean()),
           per_class_f1(y, ens_oof.argmax(1)),
           f"paired {N_REP}x{N_FOLDS}: ens {ens_sc.mean():.5f} vs cat {cat_sc.mean():.5f} (delta {d.mean():+.5f})")

  repeat 1: CatBoost 0.82051 | Ensemble 0.83631 | delta +0.01580


  repeat 2: CatBoost 0.82284 | Ensemble 0.83699 | delta +0.01416


  repeat 3: CatBoost 0.82351 | Ensemble 0.83740 | delta +0.01389


  repeat 4: CatBoost 0.82300 | Ensemble 0.83496 | delta +0.01197


  repeat 5: CatBoost 0.82291 | Ensemble 0.83779 | delta +0.01487

PAIRED repeated CV (5x5, seed=2026):
  single CatBoost : 0.82255 +/- 0.00105
  Cat+MLP ensemble: 0.83669 +/- 0.00099
  paired delta    : +0.01414 +/- 0.00127  (positive in 5/5 repeats)
  => honest gain beyond the 0.00169 noise floor? True
logged: 09_mlp_ensemble catboost_plus_mlp_pairedCV OOF macro-F1 = 0.83669


## Submission (`sub05`) — CatBoost + MLP cross-family ensemble
The first submission built from **two model families**. Recommended as the new primary private-LB pick (with `sub01` single CatBoost kept as the deterministic reference hedge).

In [7]:
name5 = "sub05_CatBoostPlusMLP_Ensemble.csv"
pred5 = ens_test.argmax(1).astype(int)
sub5 = pd.DataFrame({"id": test_ids, "sleep_stage": pred5})
assert sub5.shape == (5000, 2)
assert sub5["id"].tolist() == list(range(9000, 14000))
assert sub5["sleep_stage"].isin(CLASSES).all()
sub5.to_csv(SUB / name5, index=False)
print("wrote", SUB / name5, "| ensemble OOF macro-F1:", round(ens_macro, 5))
print("class counts:", sub5["sleep_stage"].value_counts().sort_index().to_dict())
single_test = np.load(ART / "catboost_tuned_test.npy")
diff = int((ens_test.argmax(1) != single_test.argmax(1)).sum())
print(f"sub05 vs single-CatBoost (sub01) test preds differ on {diff} of 5000 rows")

wrote C:\Users\rasul\Documents\workspace_ws\yessenov_data_lab_program\Day9\submissions\sub05_CatBoostPlusMLP_Ensemble.csv | ensemble OOF macro-F1: 0.83711
class counts: {0: 1125, 1: 1295, 2: 1273, 3: 1307}
sub05 vs single-CatBoost (sub01) test preds differ on 240 of 5000 rows


### Takeaways
- The ~0.829 wall was **not** an irreducible data limit — it was a *single-family* limit. Adding a non-tree learner (MLP) clears it.
- The MLP is both competitive on its own and **decorrelated** from CatBoost (error-corr ~0.65 vs ~0.9 tree-vs-tree), which is why the cross-family blend works where the all-tree ensembles failed.
- Honest, paired-CV gain over single CatBoost is **~+0.005 macro-F1** (well past the ~0.0017 noise floor), and it lifts the weak **Deep** class. Weights are fixed a-priori, so there is no in-sample weight overfitting.
- Remaining ceiling: Deep is symmetrically confused with Light/REM in feature space — lifting it further needs genuinely new signal, not more recombinations of `features_v1`.